In [2]:
from sklearn.model_selection import train_test_split

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 

KeyboardInterrupt: 

In [ ]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


In [ ]:
X['workclass'].value_counts()

workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
State-gov            1981
?                    1836
Self-emp-inc         1695
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB


In [ ]:
X.isna().sum()

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
dtype: int64

In [ ]:
# 1. Train = 70%, remaining = 30%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# 2. Split remaining 30% equally -> 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (34189, 14)
Validation: (7326, 14)
Test: (7327, 14)


In [ ]:
X_train['target'] = y_train
X_val['target'] = y_val
X_test['target'] = y_test

X_train.to_csv('./data/raw/train.csv')
X_val.to_csv('./data/raw/val.csv')
X_test.to_csv('./data/raw/test.csv')

In [ ]:
import numpy as np

X[categorical_features] = (
    X[categorical_features]
    .apply(lambda col: col.str.strip())
    .replace("?", np.nan)
)

/tmp/ipykernel_17502/3465442049.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[categorical_features] = (


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Example:
numeric_features = [
    "age",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)



categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
).set_output(transform='pandas')


X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [1]:
import mlflow

mlflow.set_tracking_uri(
    "http://ec2-16-16-24-206.eu-north-1.compute.amazonaws.com:5000/"
)

with mlflow.start_run():
    mlflow.log_artifact(
        "test.txt"
    )

2026/09/21 14:14:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run intrigued-dolphin-96 at: http://ec2-16-16-24-206.eu-north-1.compute.amazonaws.com:5000/#/experiments/0/runs/9a78b74f6499446abb7cfdc48da29df0.
2026/09/21 14:14:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://ec2-16-16-24-206.eu-north-1.compute.amazonaws.com:5000/#/experiments/0.


In [28]:
import pandas as pd
import mlflow
import os
import joblib
from dotenv import load_dotenv
run_id = '62d66dd219b54007b15c4bde78e3d81a'

mlflow.set_tracking_uri(
	os.getenv('MLFLOW_TRACKING_URI')
)

processor_path = mlflow.artifacts.download_artifacts(
	artifact_uri=f'runs:/{run_id}/preprocessor/preprocessor.joblib'
)

preprocessor = joblib.load(processor_path)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'education-num', 'capital-gain',
                                  'capital-loss', 'hours-per-week']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',
 

In [29]:
df = pd.read_csv('./data/raw/test.csv')
display(df.head())
df.info()
target = df['target']
df.drop(columns=['target'], inplace=True)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
0,22,State-gov,64292,Bachelors,13,Never-married,Prof-specialty,Own-child,White,Female,0,0,43,United-States,0
1,23,Private,300275,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,48,United-States,0
2,29,Private,375078,7th-8th,4,Married-civ-spouse,Other-service,Wife,White,Female,0,0,40,Mexico,0
3,46,Private,328216,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,1
4,18,Private,244571,11th,7,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7327 entries, 0 to 7326
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             7327 non-null   int64 
 1   workclass       7160 non-null   object
 2   fnlwgt          7327 non-null   int64 
 3   education       7327 non-null   object
 4   education-num   7327 non-null   int64 
 5   marital-status  7327 non-null   object
 6   occupation      7159 non-null   object
 7   relationship    7327 non-null   object
 8   race            7327 non-null   object
 9   sex             7327 non-null   object
 10  capital-gain    7327 non-null   int64 
 11  capital-loss    7327 non-null   int64 
 12  hours-per-week  7327 non-null   int64 
 13  native-country  7284 non-null   object
 14  target          7327 non-null   int64 
dtypes: int64(7), object(8)
memory usage: 858.8+ KB


In [30]:
df = pd.read_csv('./data/raw/test.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7327 entries, 0 to 7326
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             7327 non-null   int64 
 1   workclass       7160 non-null   object
 2   fnlwgt          7327 non-null   int64 
 3   education       7327 non-null   object
 4   education-num   7327 non-null   int64 
 5   marital-status  7327 non-null   object
 6   occupation      7159 non-null   object
 7   relationship    7327 non-null   object
 8   race            7327 non-null   object
 9   sex             7327 non-null   object
 10  capital-gain    7327 non-null   int64 
 11  capital-loss    7327 non-null   int64 
 12  hours-per-week  7327 non-null   int64 
 13  native-country  7284 non-null   object
 14  target          7327 non-null   int64 
dtypes: int64(7), object(8)
memory usage: 858.8+ KB


In [31]:
X_test = preprocessor.transform(df)
X_test.head()

,num__age,num__education-num,num__capital-gain,num__capital-loss,num__hours-per-week,cat__workclass_Federal-gov,cat__workclass_Local-gov,cat__workclass_Never-worked,cat__workclass_Private,cat__workclass_Self-emp-inc,...,cat__native-country_Portugal,cat__native-country_Puerto-Rico,cat__native-country_Scotland,cat__native-country_South,cat__native-country_Taiwan,cat__native-country_Thailand,cat__native-country_Trinadad&Tobago,cat__native-country_United-States,cat__native-country_Vietnam,cat__native-country_Yugoslavia
0,-1.215334,1.136011,-0.144622,-0.216649,0.205252,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.142325,-0.415009,-0.144622,-0.216649,0.609291,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.704277,-2.353784,-0.144622,-0.216649,-0.037172,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.536861,-0.415009,-0.144622,-0.216649,-0.037172,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.507366,-1.190519,-0.144622,-0.216649,-2.461408,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [32]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = mlflow.pytorch.load_model(
	f'runs:/{run_id}/model',
	map_location=device
).eval()


In [33]:
tensor = torch.tensor(
	X_test.values,
	dtype=torch.float32
)
with torch.no_grad():
    logits = model(tensor)
    probas = torch.sigmoid(logits)
    predictions = (probas >= 0.5).int()
    
    print(probas, predictions)

tensor([[0.0093],
        [0.0176],
        [0.0176],
        ...,
        [0.0502],
        [0.3980],
        [0.0016]]) tensor([[0],
        [0],
        [0],
        ...,
        [0],
        [0],
        [0]], dtype=torch.int32)


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # because EDA.ipynb is inside src/
sys.path.insert(0, str(PROJECT_ROOT))

from src.inference.predict import inference
import pandas as pd
df = pd.read_csv('./data/raw/test.csv')
inference(df)

(tensor([[0.0093],
         [0.0176],
         [0.0176],
         ...,
         [0.0502],
         [0.3980],
         [0.0016]]),
 tensor([[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]], dtype=torch.int32))

In [1]:
import pandas as pd

from src.inference.predict import predict

df = pd.read_csv(
    "./data/raw/test.csv"
)

sample = df.iloc[[0]].drop(
    columns="target"
)

result = predict(sample)

print(result)

2026-09-23 11:34:38,060 | INFO | ml_project | Inference device: cpu
INFO:ml_project:Inference device: cpu
2026-09-23 11:34:38,063 | INFO | ml_project | Loading inference artifacts from MLflow run 62d66dd219b54007b15c4bde78e3d81a
INFO:ml_project:Loading inference artifacts from MLflow run 62d66dd219b54007b15c4bde78e3d81a


2026-09-23 11:34:39,705 | INFO | ml_project | Model and preprocessor loaded successfully
INFO:ml_project:Model and preprocessor loaded successfully
2026-09-23 11:34:39,721 | INFO | ml_project | Running inference for 1 sample(s)
INFO:ml_project:Running inference for 1 sample(s)
2026-09-23 11:34:39,731 | INFO | ml_project | Inference completed successfully
INFO:ml_project:Inference completed successfully


{'probabilities': [0.00925498828291893], 'predictions': [0]}
